# EPyMARL Multi-Agent Coverage Training

Complete training pipeline for multi-agent coverage using EPyMARL + QMIX.

## Features:
- ✅ Automatic EPyMARL setup
- ✅ Checkpointing to Google Drive  
- ✅ Resumable training
- ✅ Baseline comparisons (Random, Greedy, Single-Agent)
- ✅ Visualization

---

## 1. Setup & Installation

In [ ]:
# Mount Google Drive for checkpointing
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_PATH = '/content/drive/MyDrive/epymarl_coverage'
    import os
    os.makedirs(DRIVE_PATH, exist_ok=True)
    print(f"✓ Google Drive mounted at {DRIVE_PATH}")
    USE_DRIVE = True
except:
    print("Not on Colab, using local directory")
    DRIVE_PATH = './checkpoints'
    USE_DRIVE = False

In [ ]:
%%bash
# Install dependencies
pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
pip install -q numpy scipy matplotlib pyyaml tensorboard sacred networkx gym==0.21.0

# Clone EPyMARL
if [ ! -d "epymarl" ]; then
    git clone -q https://github.com/uoe-agents/epymarl.git
    cd epymarl && pip install -q -e . && cd ..
fi

echo "✓ Installation complete!"

## 2. Upload Coverage Environment Files

Upload these files to Colab:
1. `epymarl_coverage_env.py`
2. `coverage_config.yaml`
3. `train_coverage.py`
4. `evaluate_coverage.py`

Or run this cell to download from repository:

In [ ]:
# Option: Download files from your repository
# Replace with your actual repository URL

REPO_URL = "https://raw.githubusercontent.com/YOUR_USERNAME/ind-q/main"

import requests

files = [
    'epymarl_coverage_env.py',
    'coverage_config.yaml',
    'train_coverage.py',
    'evaluate_coverage.py'
]

for filename in files:
    url = f"{REPO_URL}/{filename}"
    try:
        response = requests.get(url)
        with open(filename, 'w') as f:
            f.write(response.text)
        print(f"✓ Downloaded {filename}")
    except:
        print(f"✗ Failed to download {filename} - please upload manually")

## 3. Test Environment

In [ ]:
# Quick test of environment
from epymarl_coverage_env import CoverageEnvironment

# Create test environment
env = CoverageEnvironment(
    n_agents=2,
    grid_size=10,
    episode_limit=50,
    map_type='empty'
)

print("Environment Info:")
print(env.get_env_info())

# Test reset
obs, state = env.reset()
print(f"\nObservation shape: {obs[0].shape}")
print(f"State shape: {state.shape}")

# Test step
actions = [4, 4]  # Both agents stay
reward, done, info = env.step(actions)
print(f"\nStep result: reward={reward:.2f}, done={done}")

# Visualize
env.render()

print("\n✓ Environment test passed!")

## 4. Run Baseline Evaluation

Evaluate baselines before training to establish benchmarks.

In [ ]:
# Run baseline evaluation
!python evaluate_coverage.py \
    --n-episodes 10 \
    --n-agents 4 \
    --grid-size 20 \
    --episode-limit 200 \
    --map-type empty \
    --save-dir ./baseline_results \
    --visualize

In [ ]:
# Display baseline results
import json
from IPython.display import Image, display

with open('./baseline_results/baseline_results.json', 'r') as f:
    baseline_results = json.load(f)

print("BASELINE RESULTS:")
print("=" * 60)
for method, results in baseline_results.items():
    print(f"\n{method.upper()}:")
    print(f"  Coverage: {results['mean_coverage']:.2f}% ± {results['std_coverage']:.2f}%")
    print(f"  Return:   {results['mean_return']:.2f} ± {results['std_return']:.2f}")
    print(f"  Steps:    {results['mean_length']:.1f} ± {results['std_length']:.1f}")

print("\n" + "=" * 60)
print("TARGET: 85%+ coverage, beat all baselines")
print("=" * 60)

# Show plots
display(Image('./baseline_results/baseline_comparison.png'))
display(Image('./baseline_results/sample_episode.png'))

## 5. Training

### 5a. Initial Training (From Scratch)

In [ ]:
# Train from scratch
# This will run for ~2M steps (can take 2-4 hours on Colab)

!python train_coverage.py \
    --config ./coverage_config.yaml \
    --checkpoint-dir {DRIVE_PATH}/checkpoints \
    --results-dir {DRIVE_PATH}/results

### 5b. Resume Training (After Interruption)

In [ ]:
# Resume from latest checkpoint
!python train_coverage.py \
    --config ./coverage_config.yaml \
    --checkpoint-dir {DRIVE_PATH}/checkpoints \
    --results-dir {DRIVE_PATH}/results \
    --resume

### 5c. Monitor Training with TensorBoard

In [ ]:
# Load TensorBoard
%load_ext tensorboard
%tensorboard --logdir {DRIVE_PATH}/results/tb_logs

## 6. Evaluation

Evaluate trained agent and compare to baselines.

In [ ]:
# Evaluate trained agent
# TODO: Implement trained agent evaluation
# For now, baselines are evaluated above

print("Trained agent evaluation coming soon!")
print("Current baselines are available in baseline_results/")

## 7. Visualization & Analysis

In [ ]:
# Visualize coverage over time
import matplotlib.pyplot as plt
import numpy as np

# TODO: Load training logs and plot
# For now, show baseline comparison

from IPython.display import Image
display(Image('./baseline_results/baseline_comparison.png'))

## 8. Export Models

Download trained models for deployment.

In [ ]:
# Zip models for download
import shutil

model_dir = f"{DRIVE_PATH}/results/models"
if os.path.exists(model_dir):
    shutil.make_archive('trained_models', 'zip', model_dir)
    print("✓ Models zipped to trained_models.zip")
    print("Download from Files panel →")
else:
    print("No models found yet. Complete training first.")

---

## Next Steps

1. ✅ Run baseline evaluation (Section 4)
2. ✅ Start training (Section 5a)
3. ⏸️ Monitor with TensorBoard (Section 5c)
4. 🔄 Resume if interrupted (Section 5b)
5. 📊 Evaluate trained agent (Section 6)
6. 💾 Export models (Section 8)

**Target:** 85%+ coverage, beating all baselines!

---